# Part 3 — Sizing the Recommendations

For each recommendation: what changes in **GMV**, **profit (contribution)** and **credit loss rate** over one year.

**Method**
- **Volume = 2025** (a full year of originations).
- **Rates = seasoned loans** (originated by Jun-2025), so margins and loss rates aren't flattered by young loans.
- `loans_clean.csv` is a sample of Affirm's volume (merchants report thousands of checkouts a month; the sample
  has a few hundred loans per merchant a year), so read the $ figures as **relative** sizes.

Input: `loans_clean.csv` from Part 1.

## 0. Setup

In [ ]:
import pandas as pd
import numpy as np

loans_clean = pd.read_csv("loans_clean.csv")
loans_clean["origination_date"] = pd.to_datetime(loans_clean["origination_date"])

# Unit economics per loan (same as Part 2)
is_split = loans_clean["product_type"] == "split_pay"
term_years = np.where(is_split, 1.5, loans_clean["loan_term_months"]) / 12
loans_clean["revenue"]      = loans_clean["mdr_fees"] + loans_clean["interest_collected"]
loans_clean["funding_cost"] = loans_clean["loan_amount"] * loans_clean["cost_of_funds_bps"] / 10000 * term_years * 0.5
loans_clean["contribution"] = loans_clean["revenue"] - loans_clean["credit_loss"] - loans_clean["funding_cost"]
loans_clean["bad"] = loans_clean["loan_status"].isin(["delinquent_30", "delinquent_60", "charged_off"]).astype(int)

loans_2025 = loans_clean[loans_clean["origination_date"].dt.year == 2025]
seasoned   = loans_clean[loans_clean["origination_date"] <= "2025-06-30"]

### Baseline: one year at steady-state rates

In [ ]:
def rates(df):
    """Margin and loss rate (% of GMV) for a group of loans."""
    amount = df["loan_amount"].sum()
    return pd.Series({"margin": df["contribution"].sum() / amount,
                      "loss_rate": df["credit_loss"].sum() / amount})

vertical_rates = seasoned.groupby("vertical").apply(rates, include_groups=False)
gmv_2025 = loans_2025.groupby("vertical")["loan_amount"].sum()

baseline = pd.Series({
    "gmv":    gmv_2025.sum(),
    "profit": (gmv_2025 * vertical_rates["margin"]).sum(),
    "loss":   (gmv_2025 * vertical_rates["loss_rate"]).sum(),
})
print(f"GMV ${baseline['gmv']:,.0f} | profit ${baseline['profit']:,.0f} | loss rate {baseline['loss'] / baseline['gmv']:.2%}")
vertical_rates.round(4)

## 1. Grow healthcare & home furnishing through merchant acquisition
How much does one merchant bring in a year, and how many new merchants would +10% / +20% GMV take?
(Assumes a new merchant performs like the average existing merchant in the vertical.)

In [ ]:
per_merchant = loans_2025.groupby(["vertical", "merchant_id"]).agg(consumers=("user_id", "nunique"),
                                                                  loans=("loan_id", "count"),
                                                                  gmv=("loan_amount", "sum")).reset_index()
per_merchant["profit"] = per_merchant["gmv"] * per_merchant["vertical"].map(vertical_rates["margin"])

targets = ["healthcare", "home_furnishing"]
per_merchant[per_merchant["vertical"].isin(targets)].round(0)

In [ ]:
avg_merchant = per_merchant[per_merchant["vertical"].isin(targets)].groupby("vertical").agg(
    merchants=("merchant_id", "count"),
    consumers_per_merchant=("consumers", "mean"),
    gmv_per_merchant=("gmv", "mean"),
    profit_per_merchant=("profit", "mean"),
    vertical_gmv=("gmv", "sum"))
avg_merchant.round(0)

In [ ]:
rec1 = []
for growth in [0.10, 0.20]:
    for v, row in avg_merchant.iterrows():
        extra_gmv = row["vertical_gmv"] * growth
        new_merchants = extra_gmv / row["gmv_per_merchant"]
        rec1.append({"growth": f"+{growth:.0%}", "vertical": v,
                     "new_merchants": new_merchants,
                     "new_consumers": new_merchants * row["consumers_per_merchant"],
                     "gmv": extra_gmv,
                     "profit": extra_gmv * vertical_rates.loc[v, "margin"],
                     "loss": extra_gmv * vertical_rates.loc[v, "loss_rate"]})
rec1 = pd.DataFrame(rec1)
rec1.round(1)

**Reading it:** an average healthcare merchant brings ~57 consumers and ~$104K GMV a year (~$10K profit); an average
home furnishing merchant ~122 consumers and ~$121K GMV (~$10K profit). Each vertical has only 3–4 merchants, so
**one new merchant adds ~25–35% to the vertical** — +10% needs ~0.3–0.4 of an average merchant, +20% ~0.6–0.8.
In practice: sign **one new merchant in each vertical**. Both verticals have a loss rate (1.3–2.0%) below the
portfolio average, so this growth also *lowers* the portfolio loss rate.

## 2. Turn first-time customers into repeat customers
Two levers, both from Part 2 (1.5 and 2.5):
- **(a)** Raise the repeat rate in big-ticket verticals (healthcare, travel, home furnishing, auto parts: ~20% today).
- **(b)** Steer second loans toward interest-bearing instead of Split Pay.

In [ ]:
big_ticket = ["healthcare", "travel", "home_furnishing", "auto_parts"]
repeat_seasoned = seasoned[~seasoned["is_new_to_affirm"]]

# (a) Value of one extra repeat loan in big-ticket verticals
rb = repeat_seasoned[repeat_seasoned["vertical"].isin(big_ticket)]
repeat_loan = pd.Series({"amount": rb["loan_amount"].mean(),
                         "profit": rb["contribution"].mean(),
                         "loss":   rb["credit_loss"].mean()})
new_customers_2025 = (loans_2025["is_new_to_affirm"] & loans_2025["vertical"].isin(big_ticket)).sum()
print("New big-ticket customers in 2025:", new_customers_2025)
print("One extra repeat loan:", repeat_loan.round(1).to_dict())

In [ ]:
# (b) Difference between a repeat interest-bearing loan and a repeat Split Pay loan
by_product = repeat_seasoned.groupby("product_type")[["loan_amount", "contribution", "credit_loss"]].mean()
shift = by_product.loc["installment_ib"] - by_product.loc["split_pay"]
repeat_loans_2025 = (~loans_2025["is_new_to_affirm"]).sum()
print("Repeat loans in 2025:", repeat_loans_2025)
by_product.round(1)

In [ ]:
rec2 = []
for pts in [0.05, 0.10]:
    extra = new_customers_2025 * pts
    rec2.append({"lever": f"(a) repeat rate +{pts * 100:.0f} pts", "extra_loans": extra,
                 "gmv": extra * repeat_loan["amount"], "profit": extra * repeat_loan["profit"],
                 "loss": extra * repeat_loan["loss"]})
for pts in [0.05, 0.10]:
    moved = repeat_loans_2025 * pts
    rec2.append({"lever": f"(b) {pts:.0%} of repeat loans Split Pay -> interest-bearing", "extra_loans": 0,
                 "gmv": moved * shift["loan_amount"], "profit": moved * shift["contribution"],
                 "loss": moved * shift["credit_loss"]})
rec2 = pd.DataFrame(rec2)
rec2.round(0)

## 3. Reduce exposure to the riskiest segment: travel, FICO < 640
Part 2 (2.6) found risk concentrated in travel borrowers with FICO < 640: ~40% delinquent or charged off, and a
margin that drops to ~0% if delinquent loans charge off.

**How it would be done:** through the existing risk model — raise the approval cut-off and lower credit limits for
high-risk profiles in this segment. Here we size the outcome, not the model: **cut this segment's loan volume by
25% or 50%** (fewer approvals and/or lower amounts). Simple assumption: profit and loss shrink in proportion.

In [ ]:
def in_segment(df):
    return (df["vertical"] == "travel") & (df["fico_score"] < 640)

seg_rates = rates(seasoned[in_segment(seasoned)])
seg_gmv_2025 = loans_2025.loc[in_segment(loans_2025), "loan_amount"].sum()
print(f"Travel FICO<640 GMV in 2025: ${seg_gmv_2025:,.0f} ({seg_gmv_2025 / baseline['gmv']:.1%} of total)")
print(f"Seasoned margin {seg_rates['margin']:.1%} | loss rate {seg_rates['loss_rate']:.1%}"
      f" | bad rate {seasoned.loc[in_segment(seasoned), 'bad'].mean():.0%}")

In [ ]:
rec3 = []
for cut in [0.25, 0.50]:
    gmv = -seg_gmv_2025 * cut
    rec3.append({"action": f"travel FICO<640 volume -{cut:.0%}",
                 "gmv": gmv, "profit": gmv * seg_rates["margin"], "loss": gmv * seg_rates["loss_rate"]})
rec3 = pd.DataFrame(rec3)
rec3.round(0)

**The trade-off:** this segment is still profitable on the base loss assumption (high APR + merchant fee), so
cutting it gives up some profit in exchange for lower risk — it removes the portfolio's highest-loss pocket,
reduces loss volatility (the holiday vintage doubled travel losses) and avoids pushing more customers into
delinquency. Collections cost isn't in our margin, so these loans are less profitable than they look.
Lowering amounts rather than declining keeps customers who may become lower-risk repeat customers
(travel repeat loss: 1.2% vs 4.3% for first loans).

## 4. Reprice 0% APR
0% APR loans carry a lower merchant fee (5.0%) than interest-bearing loans at the same merchants (5.4%), though the
promo is supposed to be paid for by a higher fee (Part 2, 2.3).

In [ ]:
gmv_0pct_2025 = loans_2025.loc[loans_2025["product_type"] == "installment_0pct", "loan_amount"].sum()
rec4 = pd.DataFrame([{"action": f"0% APR merchant fee +{pts:.1%}", "gmv": 0.0,
                      "profit": gmv_0pct_2025 * pts, "loss": 0.0} for pts in [0.004, 0.01]])
print(f"0% APR GMV in 2025: ${gmv_0pct_2025:,.0f}")
rec4.round(0)

+0.4 pts brings the 0% APR fee up to the interest-bearing level; +1 pt would still be a modest ask for merchants that
choose to offer a promo. Assumes merchants keep the program (risk: some may drop it — a merchant-partnerships call).

## 5. Combined impact (one year)

In [ ]:
def combine(rows):
    total = pd.DataFrame(rows)[["gmv", "profit", "loss"]].sum()
    return pd.Series({
        "GMV change": total["gmv"] / baseline["gmv"],
        "profit change ($)": total["profit"],
        "profit change (%)": total["profit"] / baseline["profit"],
        "loss rate before": baseline["loss"] / baseline["gmv"],
        "loss rate after": (baseline["loss"] + total["loss"]) / (baseline["gmv"] + total["gmv"]),
    })

conservative = [*rec1[rec1["growth"] == "+10%"].to_dict("records"), rec2.iloc[0], rec2.iloc[2], rec3.iloc[0], rec4.iloc[0]]
ambitious    = [*rec1[rec1["growth"] == "+20%"].to_dict("records"), rec2.iloc[1], rec2.iloc[3], rec3.iloc[1], rec4.iloc[1]]

pd.DataFrame({"conservative": combine(conservative), "ambitious": combine(ambitious)}).round(4)

In [ ]:
# Contribution of each recommendation (conservative case)
parts = {
    "1. Healthcare + home furnishing +10%": rec1[rec1["growth"] == "+10%"][["gmv", "profit", "loss"]].sum(),
    "2a. Big-ticket repeat rate +5 pts":    rec2.iloc[0][["gmv", "profit", "loss"]],
    "2b. 5% of repeat loans -> IB":         rec2.iloc[2][["gmv", "profit", "loss"]],
    "3. Travel FICO<640 volume -25%":       rec3.iloc[0][["gmv", "profit", "loss"]],
    "4. 0% APR fee +0.4 pts":               rec4.iloc[0][["gmv", "profit", "loss"]],
}
pd.DataFrame(parts).T.astype(float).round(0)

## Summary
| | Conservative | Ambitious |
|---|---|---|
| Healthcare + home furnishing GMV | +10% (≈ 1 new merchant across both) | +20% (≈ 1 new merchant each) |
| Big-ticket repeat rate | +5 pts | +10 pts |
| Repeat loans moved Split Pay → interest-bearing | 5% | 10% |
| Travel FICO<640 volume (via risk model cut-off / limits) | −25% | −50% |
| 0% APR merchant fee | +0.4 pts | +1.0 pt |
| **GMV** | **+3.9%** | **+7.7%** |
| **Profit (contribution)** | **+8.3%** (~$18K on the sample) | **+17.0%** (~$36K) |
| **Credit loss rate** | **2.32% → 2.21%** | **2.32% → 2.10%** |

Profit grows faster than GMV and the loss rate falls: growth is steered toward low-loss verticals and repeat
customers, and exposure is cut where risk is concentrated.